In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

In [ ]:
#Cargamos el csv y hacemos una primera vista
df=pd.read_csv("world_bank_country_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().mean()

In [ ]:
#La deteccion de outliers esta en la correlacion

In [ ]:
#Filtramos para que solo salga paises bajos
df= df[df["country"] == "Netherlands"]

In [ ]:
#En el archivo de correlacion, nos dio que los que mas correlacion tienen con la variable son los que vamos a elegir para la regresion:
indicadores_elegidos = [ "Export value index (2015 = 100)", "Import value index (2015 = 100)", "High-technology exports (% of manufactured exports)", "Total electricity output (GWh)", "Net food imports by crop - Barley", "Machinery and transport equipment (% of value added in manufacturing)", "High-technology exports (current US$)", "Food imports (% of merchandise imports)" ]
df = df[df["indicator"].isin(indicadores_elegidos)]

In [ ]:
#cargamos y iltramos el archivo del puerto. Pasamos la fecha a datetime y extraemos el año para agrupar los kilos por año.
puerto = pd.read_csv("datosPuerto.csv", decimal=",")
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year

In [ ]:
#Al ser exportaciones, el destino es Países Bajos y el pais origen es España.
exportaciones = puerto[(puerto["PAIS_FINAL"] == "NL") & (puerto["PAIS_ORIGEN"] == "ES")]
exportaciones = exportaciones.groupby("FECHA")["PESO_KILOS"].sum()
exportaciones

In [ ]:
#Pasamos los datos de arriba, a un diccionario
kilos_exportaciones = {
2007: 582521010,
2008: 310976208,
2009: 366543925,
2010: 410957443,
2011: 754842617,
2012: 540914746,
2013: 851701657,
2014: 940298629,
2015: 595610110,
2016: 844578531,
2017: 951613421,
2018: 971474553,
2019: 1154433286,
2020: 825988465,
2021: 1040936381,
2022: 661942327,
2023: 483180404,
2024: 741719313}

In [ ]:
#Hacemos una serie con los kilos para que sea mas facil meterlo en el merge.
df_kilos = pd.Series(kilos_exportaciones).reset_index()
df_kilos.columns = ["date", "kilos"]

In [ ]:
#hacemos un pivot_table y un merge para que se junten los datos de los kilos y los indicadores
df_pivot = df.pivot_table(index="date", columns="indicator", values="value").reset_index()
df_merge = df_kilos.merge(df_pivot, on="date", how="left")
df_merge

In [ ]:
#En el caso de exportaciones, al haber tantos nan, decidimos que no imputamos los NaN ya que es un modelo bastante corto y nos quedamos hasta el año 2021
df_filtrado = df_merge[df_merge["date"] <= 2021]
df_filtrado

In [ ]:
#Al haber muchas columnas con mucha relación entre ellas, y otras con muchos valores nulos, cogemos estas como columnas.(No hace falta hacer correlacion ya que la mayoria son o columnas repetidas o llenas de Nan)
cols = ["Export value index (2015 = 100)","Import value index (2015 = 100)",'High-technology exports (current US$)',"Food imports (% of merchandise imports)",'High-technology exports (% of manufactured exports)']


In [ ]:
# Importamos librerias
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import RobustScaler

In [ ]:
#Separamos en X e y(en x ponemos las variables que hemos elegido)
X = df_filtrado[cols]
y = df_filtrado["kilos"]

In [ ]:
#Dividimos el modelo en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Escalamos nuestros datos
scaler = RobustScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

In [ ]:
#Iniciamos nuestro modelo
model = LinearRegression()
model.fit(X_train_esc, y_train)
ypred = model.predict(X_test_esc)

In [ ]:
#Miramos las métricas de error
print("MSE", mean_squared_error(y_test, ypred))
print("MAE", mean_absolute_error(y_test, ypred))
print("R²", r2_score(y_test, ypred))

In [ ]:
#Imprimimos el intercepto y los coeficientes de nuestra formula.
print("Intercept", model.intercept_)

In [ ]:
##sacamos el coeficiente de cada variable para ver cuanto influyen los indicadores en los kilos
nombres = X_train.columns
valores = model.coef_
for i in range(len(nombres)):
    print(nombres[i], valores[i])

In [ ]:
#Visualizamos la importancia de cada indicador.
plt.figure(figsize=(12,6))
plt.bar(nombres, valores, color="blue")
plt.xticks(rotation=45)
plt.ylabel("coeficientes")
plt.title("Coeficientes por variable (Exportaciones)")
plt.tight_layout()
plt.show()